# ⚡ Kronumos 8B on Google Colab (NVIDIA A100) — 100% Native Rust Engine
**Model:** `NadevA23/Kronumos` (8B Flagship Merged Safetensors)
**Engine:** Tokenectomy Rust Sub-Cortex (Zero-Token Native C-ABI Engine)
**Hardware:** NVIDIA A100 (40GB/80GB VRAM) | Precision: Native `bfloat16` + SDPA (Blazing Fast)

Notebook ini dioptimalkan khusus untuk **Google Colab A100** dengan **100% Native Rust Sub-Cortex Engine** tanpa Python mock.

### Step 1: Verifikasi GPU A100

In [ ]:
# Verifikasi GPU A100 aktif
!nvidia-smi

### Step 2: Clone Repo, Install Libs, & Unduh Engine Rust Asli

In [ ]:
# 1. Clone repository resmi Kronomus
!rm -rf Kronomus
!git clone https://github.com/Tokenectomy-Labs/Kronomus.git
%cd Kronomus

# 2. Install library inferensi terbaru
!pip install -q datasets transformers accelerate peft bitsandbytes

# 3. Unduh Native Rust Sub-Cortex Engine dari Private Repository menggunakan Token
import os
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    gh_token = None

if not gh_token:
    gh_token = os.environ.get('GITHUB_TOKEN', '')

if not gh_token:
    from getpass import getpass
    gh_token = getpass('🔑 Masukkan GitHub Token (untuk akses private Rust engine): ')
    os.environ['GITHUB_TOKEN'] = gh_token

!curl -s -H "Authorization: token {gh_token}" \
      -L "https://raw.githubusercontent.com/Tokenectomy-Labs/Tokenectomy-Sentinel/main/bin/libtokenectomy_subcortex.so" \
      -o libtokenectomy_subcortex.so

# 4. Verifikasi Engine Rust Berhasil Aktif
!python3 -c "from scripts.tokenectomy_subcortex_rust import RustSubCortex; assert RustSubCortex.is_available(), '❌ Engine Rust gagal dimuat!'; print('🎉 SUKSES: 100% Native Rust Sub-Cortex Engine AKTIF di Colab A100!')"

### Step 3: Uji Coba Cepat A100 (5 Soal Pertama, Native bfloat16 + Rust Engine)
Jalankan untuk menguji 5 soal pertama dengan mesin Rust asli.

In [ ]:
# Quick run 5 sampel dengan kecepatan penuh A100 + Rust Engine
!python scripts/kaggle_kronumos_runner.py \
    --model_id NadevA23/Kronumos \
    --dataset princeton-nlp/SWE-bench_Verified \
    --num_samples 5 \
    --bfloat16 \
    --max_turns 4 \
    --github_token "$gh_token" \
    --output_dir output_colab_5

### Step 4: Gas Full Run 500 Soal SWE-bench Verified (A100 Supercharged)
Jalankan evaluasi penuh 500 soal. Opsi `--retry_empty` memastikan jika Colab terputus, Anda bisa langsung menjalankan ulang tanpa mengulang soal yang sudah selesai.

In [ ]:
# Full 500 benchmark run pada A100
!python scripts/kaggle_kronumos_runner.py \
    --model_id NadevA23/Kronumos \
    --dataset princeton-nlp/SWE-bench_Verified \
    --num_samples 500 \
    --bfloat16 \
    --max_turns 4 \
    --retry_empty \
    --github_token "$gh_token" \
    --output_dir output_colab_500

### Step 5: Download Hasil Prediksi (predictions.jsonl)
Unduh hasil prediksi yang sudah diselaraskan oleh Sub-Cortex untuk langsung diuji di Docker.

In [ ]:
from google.colab import files
!zip -r kronumos_8b_a100_predictions.zip output_colab_500/predictions.jsonl output_colab_500/eval_metrics.json
files.download('kronumos_8b_a100_predictions.zip')